In [2]:
import os
import sys
import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report
from sklearn.datasets import make_classification
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import lightgbm as lgbm

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from hyperopt import hp

from utils import user_utils
from utils import preprocessing

In [3]:
raw_df = pd.read_csv('../data/creditcard.csv')

In [4]:
X_features, y_target = preprocessing.split_features_target(raw_df, cols= 'Time')

In [5]:
cap_X_feature = preprocessing.cap_outliers(X_features)

In [ ]:
X_train, X_test, y_train, y_test = preprocessing.data_split(cap_X_feature, y_target)
X_tr, X_val, y_tr, y_val = preprocessing.data_split(X_train, y_train, size=0.4)

NameError: name 'pp' is not defined

In [ ]:
tuner = user_utils.HyperOptTuner(max_evals=100, random_state=23)

# RandomForest 예시
catboost_search_space = {
        'iterations': hp.quniform('iterations', 100, 1000, 50),
        'depth': hp.quniform('depth', 3, 10, 1),
        'learning_rate': hp.uniform('learning_rate', 0.01, 0.03),
        'l2_leaf_reg': hp.quniform('l2_leaf_reg', 2, 30, 1),
        'border_count': hp.quniform('border_count', 32, 255, 1)
    }
dt_search_space = {
        'criterion': hp.choice('criterion', ['gini','entropy']),
        'max_depth': hp.quniform('max_depth', 3, 15, 1),
        'min_samples_split': hp.quniform('min_samples_split', 2, 20, 1),
        'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 10, 1)
    }

catboost = CatBoostClassifier()
dt_clf = DecisionTreeClassifier()

best_params, best_catboost, trials, exec_time = tuner.tune(
    catboost, X_tr, y_tr, X_val, y_val, catboost_search_space
    )
best_params, best_dt, trials, exec_time = tuner.tune(
    catboost, X_tr, y_tr, X_val, y_val, dt_search_space
    )
preprocessing.get_model_train_eval(
    best_catboost, "rf_HyperOpt", X_train, X_test, y_train, y_test, best_params
    )
preprocessing.get_model_train_eval(
    best_dt, "rf_HyperOpt", X_train, X_test, y_train, y_test, best_params
    )


CatBoostClassifier 튜닝 시작
  0%|          | 0/100 [00:00<?, ?trial/s, best loss=?]

 30%|███       | 30/100 [07:50<19:08, 16.41s/trial, best loss: -0.7911392405063291]